In [ ]:
# Import Libraries
import os
import re
from pathlib import Path

import pdfplumber
import pandas as pd


In [ ]:
# Load and Inspect PDFs
base = Path('pdfs')
files = ['FE 2024 pattern.pdf', 'FE 2025.pdf', 'BE IT CEGP011520 (28).pdf']

for name in files:
    path = base / name
    print(f'=== {name} ===')
    with pdfplumber.open(path) as pdf:
        for i, page in enumerate(pdf.pages[:3], 1):
            text = page.extract_text() or ''
            print(f'--- page {i} ---')
            print(text[:3000])
            print()


In [ ]:
# Compare PDF Layouts
from collections import Counter

for name in files:
    path = base / name
    with pdfplumber.open(path) as pdf:
        print(f'\n=== {name} ===')
        print('pages:', len(pdf.pages))
        for i, page in enumerate(pdf.pages[:2], 1):
            words = page.extract_words(x_tolerance=1, y_tolerance=2)
            print(f'page {i} words:', len(words))
            print('sample:', [w['text'] for w in words[:20]])
            print()


In [ ]:
# Build Flexible Parser for FE 2024 and 2025
from collections import OrderedDict
from dataclasses import dataclass


@dataclass
class SimpleSubject:
    code: str
    name: str
    fields: OrderedDict[str, str]


class FlexibleFEParser:
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path

    def parse(self):
        with pdfplumber.open(self.pdf_path) as pdf:
            text = '\n'.join(page.extract_text() or '' for page in pdf.pages)
        sample = text.upper()
        if 'FE 2024' in sample or '2024' in sample and 'PATTERN' in sample:
            return self._parse_fe_2024(text)
        if 'FE 2025' in sample or '2025' in sample:
            return self._parse_fe_2025(text)
        return []

    def _parse_fe_2024(self, text):
        students = []
        for chunk in re.split(r'(?=SEAT NO\.?\s*:)', text):
            if 'NAME' in chunk.upper() and 'PRN' in chunk.upper():
                students.append(chunk)
        return students

    def _parse_fe_2025(self, text):
        students = []
        for chunk in re.split(r'(?=SEAT NO\.?\s*:)', text):
            if 'NAME' in chunk.upper() and 'PRN' in chunk.upper():
                students.append(chunk)
        return students


for name in ['FE 2024 pattern.pdf', 'FE 2025.pdf']:
    parser = FlexibleFEParser(base / name)
    result = parser.parse()
    print(name, 'chunks:', len(result))
    if result:
        print(result[0][:1500])
        print()


In [ ]:
# Preserve BE IT Extraction Logic
# The BE IT parser remains unchanged; the FE additions are isolated in a detector-based branch.
print('Notebook ready to inspect FE-specific layouts while keeping the BE path intact.')

In [ ]:
# Validate Extraction Results
# This notebook serves as an exploratory scaffold. The actual parser integration will be verified by running main.py after the code changes.
print('Validation scaffold prepared.')